In [1]:
import argparse
import pickle
import os
from pathlib import Path
import torch
import tqdm
from torch.utils.data import DataLoader
from torch.optim import Adam
import gc
from models.bert import BERT
from models.tokenizer import AsmTokenizer
from normalize_instr import normalize_instruction

data_dir = "."

In [ ]:
import re

# New tokenization regex (compatible with semantically enhanced tokens)
token_pattern = r"""
    \<[A-Za-z]+:[A-Za-z0-9_]+(?::\w+)?\> |  # Match <TYPE:value> or <TYPE:subtype:value>
    \<[A-Z]+\> |                            # Match simple all-uppercase tags <TARGET>
    \[[^\]]+\] |                            # Match entire memory expressions [content]
    [A-Za-z]+:[A-Za-z0-9_]+ |               # Match category:opcode (e.g., data_transfer:mov)
    [\[\],:;*+./-] |                        # Match single special characters
    [\w]+                                   # Match regular word characters
"""

def tokenize_assembly(assembly_str):
    # 使用详细模式忽略空白和注释
    tokens = re.findall(token_pattern, assembly_str, re.VERBOSE)
    return [token.strip() for token in tokens if token.strip()]
sample_instruction = (
    "data_transfer:mov <REG:gpr>, [<BASE:gpr>+<INDEX:gpr:4>+<DISP:small>]"
)
def expand_memory_tokens(tokens):
    expanded = []
    for token in tokens:
        if token.startswith('[') and token.endswith(']'):
            # 拆解内存表达式
            inner = token[1:-1]
            if inner:
                expanded.append('[')
                # 按+拆分但保留操作符
                parts = re.split(r'(\+)', inner)
                expanded.extend([p for p in parts if p])
                expanded.append(']')
            else:
                expanded.append(token)
        else:
            expanded.append(token)
    return expanded
# 分词结果
tokens = tokenize_assembly(sample_instruction)
expanded = expand_memory_tokens(tokens)
print(expanded)

['data_transfer', ':', 'mov', '<REG:gpr>', ',', '[', '<BASE:gpr>', '+', '<INDEX:gpr:4>', '+', '<DISP:small>', ']']


In [2]:
import re
import os
from datasets import load_dataset
vocab = {"<PAD>": 0, "<CLS>": 1, "<SEP>": 2, "<MASK>": 3, "<UNK>": 4, "<const>": 5}
for dataset_name in ["baseline-train", "baseline-val", "baseline-test"]:
    dataset_path = os.path.join(".", "outputs", f"{dataset_name}.jsonl")
    dataset = load_dataset('json', data_files=dataset_path, split="train", streaming=True)
    for data in dataset:
        for instrs in data['instruction_blocks']:
            tokens = re.findall(r"<[A-Z]+>|[\w]+|[\[\],:;*+./-]", instrs)
            for token in tokens:
                if token not in vocab:
                    vocab[token] = len(vocab)

In [5]:
def save_vocab(filepath):
    with open(filepath, "w") as f:
        for token, idx in sorted(vocab.items(), key=lambda x: x[1]):
            f.write(f"{token}\n")
    print(f"Vocab saved to {filepath}")
save_vocab(os.path.join(".", "outputs", "baseline-vocab.txt"))

Vocab saved to .\outputs\baseline-vocab.txt


In [5]:
with open(os.path.join(data_dir, "outputs", "baseline-vocab.txt"), "r") as f:
    lines = f.readlines()
lines = [line.strip() for line in lines if line.strip()]
new_lines = []
for line in lines:
    if re.match(r"0x[0-9a-fA-F]+", line) or re.match(r"[-+]?\b\d+\b", line):
        continue
    else:
        new_lines.append(line)
len(new_lines)

428

In [4]:
import pandas as pd
results_string=pd.Categorical(results_string)
results_string

['test rdi, rdi', 'mov rax, qword ptr fs:[0x28]', 'call 0x162efd0', 'lea r14d, [rax - 1]', 'jae 0x6a2105', ..., 'js 0x44f58f', 'jne 0x44f503', 'mov ebp, 0', 'call 0x44ed50', 'cmp r12d, -1']
Length: 140721658
Categories (25726565, object): ['0x403d80:\tsub\trsp, 8', '0x403d84:\tmov\trax, qword ptr [rip + 0x2a4265]', '0x403d8b:\ttest\trax, rax', '0x403d8e:\tje\t0x403d92', ..., 'xorps xmm8, xmm8', 'xorps xmm9, xmm9', 'xrstor ptr [rsp + 0x40]', 'xsave ptr [rsp + 0x40]']

In [ ]:
train_data = load_assembly_data(data)
del data
gc.collect()

In [3]:
import pandas as pd
train_data = pd.DataFrame(train_data, columns=["instr1", "instr2"])
train_data.info(memory_usage="deep")
train_data['instr1'] = train_data["instr1"].astype("category")
train_data['instr2'] = train_data["instr2"].astype("category")
train_data.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69703068 entries, 0 to 69703067
Data columns (total 2 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   instr1  object
 1   instr2  object
dtypes: object(2)
memory usage: 9.7 GB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69703068 entries, 0 to 69703067
Data columns (total 2 columns):
 #   Column  Dtype   
---  ------  -----   
 0   instr1  category
 1   instr2  category
dtypes: category(2)
memory usage: 3.5 GB


In [ ]:
import pandas as pd
train_data = pd.read_csv("./outputs/train_data.csv")

In [8]:
t1,t2=train_data.iloc[0]
t1

'call 0xd138bc'

In [6]:
len(train_data)

69703068

In [3]:
with open("train_data.txt", "w") as f:
    for i in train_data:
        f.write(f"{i[0]}\n{i[1]}\n")

In [4]:
binary = os.path.join(data_dir, "outputs", f"baseline-valid.pkl")
with open(binary, 'rb') as c:
    data = pickle.load(c)

valid_data = load_assembly_data(data)

In [5]:
with open("valid_data.txt", "w") as f:
    for i in valid_data:
        f.write(f"{i[0]}\n{i[1]}\n")